# 07 BLP Demand Estimation: Random-Coefficients Logit and Share Inversion

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/blob/main/05-Micro-Models/07_BLP_Demand_Estimation.ipynb) [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/main?filepath=05-Micro-Models/07_BLP_Demand_Estimation.ipynb) [![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)
## The Lens: Recovering Willingness to Pay from Market Shares
Product-level market shares compress millions of household choices into a few equilibrium outcomes. Structural demand estimation asks whether those outcomes can be inverted to recover latent mean utilities and, eventually, substitution patterns and willingness to pay. The Berry-Levinsohn-Pakes (BLP) framework extends multinomial logit by allowing heterogeneous tastes, which prevents every product from being an equally good substitute after conditioning on mean utility.

The computational core is a fixed point. Observed shares are held fixed while the mean utility vector $\delta$ is updated until simulated shares match them. That contraction is only one layer of the estimator: a production BLP analysis also instruments endogenous prices and searches over nonlinear taste parameters in an outer GMM objective. This notebook isolates the inner inversion, simulates data where the truth is known, verifies the share residual, and makes the nested structure explicit so the econometric identification problem is never confused with numerical convergence.

**Economic question.** In *07 BLP Demand Estimation: Random-Coefficients Logit and Share Inversion*, what must remain economically invariant when the computational representation changes? Microeconomic computation makes equilibrium and incentive constraints operational. The useful question is how preferences, technologies, information, or strategic beliefs map into choices and welfare, and whether the computed solution respects feasibility, optimality, and equilibrium conditions. Whenever multiple equilibria or corner solutions are possible, the numerical method should expose rather than hide that economic structure.

### Learning Objectives
- **Derive** the logit share equation and Berry share-inversion contraction.
- **Simulate** random-coefficient market shares with common consumer draws.
- **Recover** mean utilities by fixed-point iteration and diagnose convergence.
- **Distinguish** the inner contraction from the outer IV/GMM identification problem.

### Prerequisites
- `04_Discrete_Choice_Models.ipynb`: logit choice probabilities and random utility.
- `../03-Economic-Modeling/07_Structural_Estimation.ipynb`: structural estimation and nested fixed points.
- `../06-Econometrics/05_Instrumental_Variables.ipynb`: price endogeneity and instruments.
* **Learning-path prerequisite:** [`06_Information_Economics.ipynb`](06_Information_Economics.ipynb)


> **Learning path:** Building on [`06_Information_Economics.ipynb`](06_Information_Economics.ipynb); this notebook closes the current track.


## Table of Contents

1. [From random utility to market shares](#random-utility)
2. [Berry's contraction](#berry-contraction)
3. [Synthetic random-coefficients market](#synthetic-market)
4. [Inversion diagnostics](#inversion-diagnostics)
5. [From inversion to BLP GMM](#blp-gmm)
6. [Exercises](#exercises)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"font.size": 12, "figure.figsize": (10, 6), "figure.dpi": 120})
np.set_printoptions(suppress=True, precision=5, linewidth=120)

from scipy.special import logsumexp


<a id="random-utility"></a>
## 1. From Random Utility to Market Shares

Consumer $i$ in market $t$ chooses product $j$ when

$$u_{ijt}=\delta_{jt}+\sigma x_{jt}\nu_i+\varepsilon_{ijt}$$

is maximal, where $\varepsilon$ is type-I extreme value and $\nu_i\sim N(0,1)$ creates taste heterogeneity. Conditional on $\nu_i$, the logit probability is

$$P_{ijt}=\frac{\exp(\delta_{jt}+\sigma x_{jt}\nu_i)}{1+\sum_{k=1}^J\exp(\delta_{kt}+\sigma x_{kt}\nu_i)}.$$

Integrating over $\nu$ gives simulated market shares. The outside good has normalized mean utility zero, so its share disciplines the absolute level of $\delta$.


<a id="berry-contraction"></a>
## 2. Berry's Contraction

For a trial mean utility vector $\delta^h$, simulate shares $s(\delta^h;\sigma)$. The BLP inversion updates

$$\delta^{h+1}_{jt}=\delta^h_{jt}+\log s^{obs}_{jt}-\log s_{jt}(\delta^h;\sigma).$$

At a fixed point, simulated and observed shares coincide. Numerically, monitor the **share residual** as well as the update norm. The contraction solves for $\delta$ conditional on nonlinear taste parameters; it does not solve the endogeneity of price.


<a id="synthetic-market"></a>
## 3. Synthetic Random-Coefficients Market

We generate many independent markets with known mean utilities and a common set of simulation draws. Reusing draws makes the contraction deterministic and prevents Monte Carlo noise from masquerading as non-convergence.


In [ ]:
def simulated_shares(delta, x_random, sigma, draws):
    """Simulate BLP inside-good shares for one market."""
    delta = np.asarray(delta, dtype=float)
    x_random = np.asarray(x_random, dtype=float)
    draws = np.asarray(draws, dtype=float)
    utilities = delta[:, None] + sigma * x_random[:, None] * draws[None, :]
    # Include outside-good utility 0 with a stable log denominator.
    log_denom = logsumexp(np.vstack([np.zeros(draws.size), utilities]), axis=0)
    probabilities = np.exp(utilities - log_denom)
    return probabilities.mean(axis=1)


def berry_inversion(observed, x_random, sigma, draws, tol=1e-12, max_iter=10_000):
    """Recover mean utilities from observed shares by Berry contraction."""
    observed = np.asarray(observed, dtype=float)
    if np.any(observed <= 0) or observed.sum() >= 1:
        raise ValueError("Inside shares must be positive and sum to less than one.")
    outside = 1.0 - observed.sum()
    delta = np.log(observed) - np.log(outside)  # simple-logit starting value
    for iteration in range(1, max_iter + 1):
        predicted = np.clip(simulated_shares(delta, x_random, sigma, draws), 1e-300, 1.0)
        delta_new = delta + np.log(observed) - np.log(predicted)
        update = float(np.max(np.abs(delta_new - delta)))
        delta = delta_new
        if update < tol:
            break
    else:
        raise RuntimeError(f"BLP contraction did not converge; update={update:.3e}")
    predicted = simulated_shares(delta, x_random, sigma, draws)
    return delta, iteration, update, float(np.max(np.abs(predicted - observed)))

n_markets, n_products, n_draws = 20, 5, 6_000
beta0, beta_x, alpha_price, sigma_true = -1.0, 1.2, 1.0, 0.8
draws = rng.normal(size=n_draws)
records = []
for market in range(n_markets):
    quality = rng.normal(size=n_products)
    price = 2.0 + 0.5 * quality + rng.normal(scale=0.35, size=n_products)
    xi = rng.normal(scale=0.15, size=n_products)
    delta_true = beta0 + beta_x * quality - alpha_price * price + xi
    shares = simulated_shares(delta_true, quality, sigma_true, draws)
    for j in range(n_products):
        records.append((market, j, quality[j], price[j], delta_true[j], shares[j]))

market_data = pd.DataFrame(records, columns=["market", "product", "quality", "price", "delta_true", "share"])
market_data.head()


<a id="inversion-diagnostics"></a>
## 4. Inversion Diagnostics

Because the data were generated by the same model used in inversion, recovered mean utilities should match the latent truth up to numerical tolerance. In empirical work the truth is unavailable, so the observable diagnostic is the maximum share discrepancy together with fixed-point stability under tighter tolerances and more simulation draws.


In [ ]:
results = []
for market, group in market_data.groupby("market", sort=True):
    recovered, iterations, update, share_error = berry_inversion(
        group["share"].to_numpy(),
        group["quality"].to_numpy(),
        sigma_true,
        draws,
    )
    delta_error = float(np.max(np.abs(recovered - group["delta_true"].to_numpy())))
    results.append((market, iterations, update, share_error, delta_error))

diagnostics = pd.DataFrame(results, columns=["market", "iterations", "update", "share_error", "delta_error"])
display(diagnostics.describe().T)
assert diagnostics["share_error"].max() < 1e-9
assert diagnostics["delta_error"].max() < 1e-8


In [ ]:
sample = market_data.query("market == 0").copy()
sample["delta_recovered"], *_ = berry_inversion(sample["share"], sample["quality"], sigma_true, draws)
fig, ax = plt.subplots()
ax.scatter(sample["delta_true"], sample["delta_recovered"], s=70)
lo, hi = sample[["delta_true", "delta_recovered"]].to_numpy().min(), sample[["delta_true", "delta_recovered"]].to_numpy().max()
ax.plot([lo, hi], [lo, hi], "--", label="45° line")
ax.set(xlabel="true mean utility", ylabel="recovered mean utility", title="Berry inversion recovers the synthetic truth")
ax.legend()
plt.show()


<a id="blp-gmm"></a>
## 5. From Share Inversion to BLP GMM

The full BLP estimator nests the contraction inside an outer objective. If prices are endogenous, mean utility is decomposed as

$$\delta_{jt}=x_{jt}'\beta-\alpha p_{jt}+\xi_{jt},$$

and instruments $Z$ satisfy $E[Z'\xi]=0$. For each candidate nonlinear parameter $\sigma$:

1. invert shares to obtain $\delta(\sigma)$;
2. estimate linear parameters $(\beta,\alpha)$ by IV/GMM;
3. recover $\xi(\sigma)$;
4. evaluate

$$Q(\sigma)=g(\sigma)'Wg(\sigma),\qquad g(\sigma)=\frac{1}{N}Z'\xi(\sigma);$$

5. optimize over $\sigma$ and then compute robust uncertainty.

Keeping the contraction and IV/GMM layers separate makes failures diagnosable: share mismatch is numerical; invalid instruments are an identification failure.


## Key Equations

These relations are collected from the derivations above as a review map. Their assumptions and derivations remain part of the result; this box is not a substitute for them.

**1. Core relation**

$$u_{ijt}=\delta_{jt}+\sigma x_{jt}\nu_i+\varepsilon_{ijt}$$

**2. Core relation**

$$P_{ijt}=\frac{\exp(\delta_{jt}+\sigma x_{jt}\nu_i)}{1+\sum_{k=1}^J\exp(\delta_{kt}+\sigma x_{kt}\nu_i)}.$$

**3. Core relation**

$$\delta^{h+1}_{jt}=\delta^h_{jt}+\log s^{obs}_{jt}-\log s_{jt}(\delta^h;\sigma).$$

**4. Core relation**

$$\delta_{jt}=x_{jt}'\beta-\alpha p_{jt}+\xi_{jt},$$


## Exercises

**1. IIA versus random coefficients (Conceptual):** Explain why the simple logit model implies proportional substitution and how a random coefficient on product quality changes cross-price substitution patterns.

**2. Numerical inversion (Applied):** Repeat the experiment for `sigma` in `{0, 0.4, 0.8, 1.5}`. Record iteration counts and verify share errors. Explain why a harder substitution pattern can change contraction speed.

**3. Endogenous prices (Challenge):** Generate price using an unobserved cost shock correlated with `xi`, add an excluded cost shifter as an instrument, and implement the outer IV/GMM step. Compare OLS and IV estimates of the price coefficient.


## Summary & Key Takeaways

- Random coefficients relax the simple-logit substitution pattern by allowing heterogeneous tastes.
- Berry's contraction recovers mean utilities by matching observed and simulated shares.
- Share inversion is a numerical fixed point; price endogeneity is a separate econometric identification problem.
- Reusing simulation draws and reporting share residuals make the inner loop reproducible and auditable.


## References & Further Reading

- Berry, S. (1994). Estimating discrete-choice models of product differentiation. *RAND Journal of Economics*, 25(2), 242–262.
- Berry, S., Levinsohn, J. & Pakes, A. (1995). Automobile prices in market equilibrium. *Econometrica*, 63(4), 841–890.
- Nevo, A. (2000). A practitioner's guide to estimation of random-coefficients logit models of demand. *Journal of Economics & Management Strategy*, 9(4), 513–548.
